In [1]:
import timm
import torch
import torch.nn as nn
from torch.optim import AdamW
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd

In [2]:
class MobileNetV4(nn.Module):
    def __init__(self, num_classes: int = 4, in_chans: int = 1, pretrained: bool = False, drop_rate: float = 0.1, drop_path_rate: float = 0.1):
        super().__init__()
        
        self.model = timm.create_model(
            "mobilenetv4_conv_small_050",
            pretrained=pretrained,
            in_chans=in_chans,
            num_classes=num_classes,
            drop_rate=drop_rate,
            drop_path_rate=drop_path_rate,
        )

    def forward(self, x):
        return self.model(x)

In [3]:
class CWRUDataset(Dataset):
    def __init__(self, path):
        self.dataset = torch.load(path)
        self.data    = self.dataset['data']
        self.label   = self.dataset['label']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image = self.data[index].float() / 255.0
        label = self.label[index].long()
        return image, label

In [4]:
class Trainer:
    def __init__(self, model: MobileNetV4, train_path: str, val_path: str, num_classes: int=4,
                 batch_size: int=32, lr: float=1e-3, weight_decay: float=1e-5, max_epochs: int=100,
                 patience: int=10, min_delta: float=1e-4):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        self.model = model.to(self.device)

        if self.device == "cuda" and torch.cuda.device_count() > 1:
            print(f"Using DataParallel across {torch.cuda.device_count()} GPUs")
            self.model = nn.DataParallel(self.model)
        
        self.train_loader = DataLoader(CWRUDataset(train_path), batch_size=batch_size, shuffle=True)
        self.val_loader = DataLoader(CWRUDataset(val_path), batch_size=batch_size, shuffle=False)
        self.max_epochs = max_epochs
        self.optimizer = AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.CrossEntropyLoss()

        self.patience = patience
        self.min_delta = min_delta
        self._es_counter = 0
        self._es_prev = float('inf')

    
    def _check_early_stop(self, val_loss: float) -> bool:
        if val_loss < self._es_prev - self.min_delta:
            self._es_counter = 0
            self._es_prev = val_loss
        else:
            self._es_counter += 1
        return self._es_counter >= self.patience

    
    def _train_epoch(self, loader, train):
        self.model.train() if train else self.model.eval()
        context = torch.enable_grad() if train else torch.no_grad()
        total_loss = correct = total = 0.0
        with context:
            for imgs, labels in loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                if train:
                    self.optimizer.zero_grad()
    
                out = self.model(imgs)
                loss = self.criterion(out, labels)
                if train:
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item() * imgs.size(0)
                correct += (out.argmax(1) == labels).sum().item()
                total += imgs.size(0)
        return total_loss / total, correct / total

    
    def train(self, save_path: str = 'best_model.pt', history_path: str = 'history.csv'):
        best_val_loss = float('inf')
        history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
        header = f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>8} | {'':>18}"
        divider = '-' * len(header)
        print(f"\nTraining for up to {self.max_epochs} epochs on {self.device}")
        print(f"Early stopping: patience={self.patience}, min_delta={self.min_delta}")
        print(divider)
        print(header)
        print(divider)
    
        stopped_early = False
        for epoch in range(1, self.max_epochs + 1):
            train_loss, train_acc = self._train_epoch(self.train_loader, train=True)
            val_loss, val_acc = self._train_epoch(self.val_loader, train=False)
    
            history['epoch'].append(epoch)
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            is_best = val_loss < best_val_loss
            should_stop = self._check_early_stop(val_loss)
            patience_remaining = self.patience - self._es_counter
            
            tag = f"best | patience: {self.patience}/{self.patience}" if is_best else \
                  f"patience: {patience_remaining}/{self.patience}"

            print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>8.4f} | {val_acc:>8.2%} | {tag}")
    
            if is_best:
                best_val_loss = val_loss
                core = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
                torch.save(core.state_dict(), save_path)

            if should_stop:
                print(divider)
                print(f"Early stopping triggered after {epoch} epochs ({self.patience} epochs without improvement).")
                stopped_early = True
                break
    
        print(divider)
        reason = "Early stopping" if stopped_early else "Training complete"
        print(f"{reason}. Best val loss: {best_val_loss:.4f} — model saved to '{save_path}'")
    
        self.history_df = pd.DataFrame(history)
        self.history_df.to_csv(history_path, index=False)
        print(f"History saved to '{history_path}'\n")


    def evaluate(self, loader=None, load_path: str = None) -> dict:
        if load_path:
            state = torch.load(load_path, map_location=self.device)
            core  = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
            core.load_state_dict(state)
            print(f"Loaded weights from '{load_path}'")
    
        if loader is None:
            loader = self.val_loader
    
        self.model.eval()
        all_labels, all_preds, all_probs = [], [], []
    
        with torch.no_grad():
            for imgs, labels in loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                logits = self.model(imgs)
                probs = torch.softmax(logits, dim=1)
    
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs.argmax(dim=1).cpu().numpy())
                all_probs.append(probs.cpu().numpy())
    
        y_true  = np.concatenate(all_labels)
        y_pred  = np.concatenate(all_preds)
        y_probs = np.concatenate(all_probs)
    
        num_classes = y_probs.shape[1]
        multiclass  = num_classes > 2
        avg         = "macro" if multiclass else "binary"
    
        metrics = {
            "accuracy":  accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
            "recall":    recall_score(y_true, y_pred, average=avg, zero_division=0),
            "f1":        f1_score(y_true, y_pred, average=avg, zero_division=0),
            "auc_roc":   roc_auc_score(
                             y_true,
                             y_probs if multiclass else y_probs[:, 1],
                             multi_class="ovr" if multiclass else "raise",
                             average=avg,
                         ),
            "confusion_matrix": confusion_matrix(y_true, y_pred),
        }
    
        width = 45
        print("\n" + "=" * width)
        print(f"{'EVALUATION RESULTS':^{width}}")
        print("=" * width)
        print(f"  {'Accuracy' :<12} {metrics['accuracy'] :>8.4f}")
        print(f"  {'Precision':<12} {metrics['precision']:>8.4f}  ({avg})")
        print(f"  {'Recall'   :<12} {metrics['recall']   :>8.4f}  ({avg})")
        print(f"  {'F1 Score' :<12} {metrics['f1']       :>8.4f}  ({avg})")
        print(f"  {'AUC-ROC'  :<12} {metrics['auc_roc']  :>8.4f}  ({avg})")
        print("-" * width)
        print("Confusion Matrix:")
        print(metrics["confusion_matrix"])
        print("-" * width)
        print("Classification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))
        print("=" * width + "\n")
    
        return metrics

In [5]:
trainer = Trainer(
    model = MobileNetV4(num_classes=4, in_chans=1),
    train_path=r"/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/train_synthetic.pt",
    val_path=r"/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/val.pt",
    num_classes=4,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=50
)

Using DataParallel across 2 GPUs


In [ ]:
trainer.train()


Training for up to 50 epochs on cuda
Early stopping: patience=10, min_delta=0.0001
--------------------------------------------------------------------------
 Epoch | Train Loss | Train Acc | Val Loss |  Val Acc |                   
--------------------------------------------------------------------------
     1 |     1.0438 |   79.82% |   0.1446 |   94.38% | best | patience: 10/10
     2 |     0.1939 |   95.02% |   0.3354 |   93.66% | patience: 9/10
     3 |     0.1074 |   97.09% |   0.0670 |   97.50% | best | patience: 10/10
     4 |     0.0895 |   98.12% |   0.0923 |   97.44% | patience: 9/10
     5 |     0.1194 |   97.59% |   0.0718 |   97.74% | patience: 8/10
     6 |     0.0736 |   98.39% |   0.1863 |   95.18% | patience: 7/10
     7 |     0.0383 |   99.21% |   0.1656 |   94.09% | patience: 6/10
     8 |     0.0445 |   99.05% |   0.9462 |   88.50% | patience: 5/10
     9 |     0.0412 |   99.08% |   0.1608 |   95.02% | patience: 4/10
    10 |     0.0321 |   99.40% |   0.0145 |  

In [ ]:
test_loader = DataLoader(CWRUDataset("/kaggle/input/datasets/lewisdo/train-cnn/Scalogram/test.pt"),
                         batch_size=32, shuffle=False)

metrics = trainer.evaluate(loader=test_loader, load_path="/kaggle/working/best_model.pt")